# 20 — Lifecycle Feature Extraction

Extract 8 scalar health indicators from every run of one sensor channel, then plot the degradation trajectory over time.

**Dataset**: XJTU-SY Bearing — Bearing 1_1 (35 Hz / 12 kN, 123 run-to-failure snapshots)  
**API**: `assay.lifecycle_features()` · `assay.plot_lifecycle()`

In [1]:
import warnings, logging, sys
from pathlib import Path

warnings.filterwarnings("ignore")
logging.getLogger("isa_phm").setLevel(logging.ERROR)

WRAPPER_ROOT = Path("..").resolve()
if str(WRAPPER_ROOT) not in sys.path:
    sys.path.insert(0, str(WRAPPER_ROOT))

from isa_phm import ISAWrapper
from bokeh.io import output_notebook
from bokeh.plotting import show as bokeh_show
output_notebook()

ISA_JSON = Path(r"G:\ISA\Datasets\XJTU-SY_Bearing_Datasets\XJTU-SY_Bearing_Datasets\XJTU-SY Bearing Datasets-ISA-PHM-Out.json")
print("Exists:", ISA_JSON.exists())

Loading BokehJS ...

Exists: True


In [2]:
wrapper = ISAWrapper(ISA_JSON, strict_validation=False, cache_maxsize=10)

study = wrapper.study("Bearing 1_1")
assay = study.assay(1)          # 1 = horizontal accelerometer (integer index)

print(f"Study : {study.title}")
print(f"Assay : {assay.assay_id}  ({assay.run_count} runs)")

Study : Bearing 1_1
Assay : a_st01_se01  (123 runs)


## 1. Extract lifecycle features

`lifecycle_features()` computes 8 scalar statistics per run using a thread-pool.
The result is a tidy DataFrame — one row per run.

| Feature | Sensitivity |
|---------|-------------|
| `rms` | Overall energy — rises steadily with fatigue |
| `kurtosis` | Impulsiveness — spikes sharply near failure |
| `crest_factor` | Peak/RMS — amplifies early micro-pitting |
| `peak2peak` | Dynamic range |
| `std` | Signal variability |
| `skewness` | Asymmetry |
| `mean` | DC offset |
| `max` | Absolute maximum |

In [4]:
import pandas as pd

lc = assay.lifecycle_features(file_type="raw", n_workers=8)

print(f"Shape: {lc.shape}")
display(lc.head(8))

Shape: (123, 17)


,run_id,run_number,study_id,assay_id,rms,max,mean,peak2peak,kurtosis,std,crest_factor,skewness,fv_Fault Type,fv_Bearing Lifetime,fv_Motor speed,fv_Pressure Axial,fv_Pressure Radial
0,run_001,1,12a95728-7b62-4f7f-8c49-82b4512fe5d2,a_st01_se01,0.563890,2.354336,-0.007343,4.884219,0.071352,0.563842,4.486485,-0.001797,Outer Race,123 min,2100 RPM,12 kN,12 kN
1,run_002,2,12a95728-7b62-4f7f-8c49-82b4512fe5d2,a_st01_se01,0.589078,3.195643,-0.007776,6.818748,0.137942,0.589026,6.150471,-0.014991,Outer Race,122 min,2100 RPM,12 kN,12 kN
2,run_003,3,12a95728-7b62-4f7f-8c49-82b4512fe5d2,a_st01_se01,0.589536,3.321779,-0.001499,6.517446,0.246259,0.589534,5.634563,0.022943,Outer Race,121 min,2100 RPM,12 kN,12 kN
3,run_004,4,12a95728-7b62-4f7f-8c49-82b4512fe5d2,a_st01_se01,0.597274,2.872622,0.006744,5.412161,0.248190,0.597236,4.809554,0.013573,Outer Race,120 min,2100 RPM,12 kN,12 kN
4,run_005,5,12a95728-7b62-4f7f-8c49-82b4512fe5d2,a_st01_se01,0.604645,4.136920,-0.012305,7.663488,0.393918,0.604520,6.841896,0.036585,Outer Race,119 min,2100 RPM,12 kN,12 kN
5,run_006,6,12a95728-7b62-4f7f-8c49-82b4512fe5d2,a_st01_se01,0.627667,3.986502,-0.009535,8.613884,0.550157,0.627595,7.372348,0.024117,Outer Race,118 min,2100 RPM,12 kN,12 kN
6,run_007,7,12a95728-7b62-4f7f-8c49-82b4512fe5d2,a_st01_se01,0.639083,3.711307,-0.017390,8.586109,0.821914,0.638846,7.627805,0.008003,Outer Race,117 min,2100 RPM,12 kN,12 kN
7,run_008,8,12a95728-7b62-4f7f-8c49-82b4512fe5d2,a_st01_se01,0.628535,2.968681,-0.034927,7.254839,0.576180,0.627564,6.819285,-0.025558,Outer Race,116 min,2100 RPM,12 kN,12 kN


## 2. RMS degradation trajectory

RMS is the most reliable early-degradation indicator for rolling-element bearings.

In [5]:
fig = assay.plot_lifecycle(feature="rms", file_type="raw")
bokeh_show(fig)

## 3. Kurtosis — early fault indicator

Kurtosis rises well before RMS because it measures impulsiveness.  
The sharp early spike (then gradual rise) is the classic bearing fault pattern.

In [6]:
fig = assay.plot_lifecycle(feature="kurtosis", file_type="raw")
bokeh_show(fig)

## 4. Crest factor — micro-pitting sensitivity

Crest factor (peak / RMS) amplifies very early surface damage.  
Useful as a complementary feature alongside RMS.

In [7]:
fig = assay.plot_lifecycle(feature="crest_factor", file_type="raw")
bokeh_show(fig)

## 5. Plot all other features

Plot any feature from the lifecycle DataFrame with `plot_lifecycle()`.

In [8]:
for feature in ["std", "peak2peak", "skewness", "max"]:
    fig = assay.plot_lifecycle(feature=feature, file_type="raw")
    bokeh_show(fig)